## 1. Install and Import Libraries

In [1]:
!pip install -q openai pillow




[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import datetime
import json
import base64

from google import genai
from google.genai import types
from PIL import Image

## 2. Define Configuration Variables

In [3]:
# API credentials
# Set GEMINI_API_KEY in your environment before running this notebook.
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

# Model configuration
GEMINI_MODEL = "gemini-3.6-flash"

# Interface URL associated with this evaluation
INTERFACE_URL = "https://www.amazon.com.br/s?k=iphone"

# API client
from google import genai

gemini_client = genai.Client(api_key=GEMINI_API_KEY)


## 3. Extract Interface Screenshots

In [4]:
def image_to_base64(image_path):
    """Convert an image file to a Base64-encoded string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def resize_image(image_path, max_width=1280):
    """Resize an image when necessary to meet API image-size limits."""
    image = Image.open(image_path)

    if image.width > max_width:
        ratio = max_width / image.width
        new_size = (max_width, int(image.height * ratio))
        image = image.resize(new_size, Image.LANCZOS)
        image.save(image_path)
        print(f"[✓] Image resized to {new_size}")
    else:
        print(f"[✓] Image is ready: {image.width}x{image.height} px")




In [5]:
# Directory containing the UI screenshots.
SCREENSHOT_DIR = "./inputs"

# Supported image extensions
VALID_EXTENSIONS = (".png", ".jpg", ".jpeg", ".webp")

# List image files in alphabetical order.
ui_files = sorted([
    os.path.join(SCREENSHOT_DIR, filename)
    for filename in os.listdir(SCREENSHOT_DIR)
    if filename.lower().endswith(VALID_EXTENSIONS)
])

if not ui_files:
    raise FileNotFoundError(f"No screenshots found in: {SCREENSHOT_DIR}")

print(f"[✓] Found {len(ui_files)} screenshot(s):")
for image_path in ui_files:
    print(f"    • {os.path.basename(image_path)}")

[✓] Found 12 screenshot(s):
    • 01.png
    • 02.png
    • 03.png
    • 04.png
    • 05.png
    • 06.png
    • 07.png
    • 08.png
    • 09.png
    • 10.png
    • 11.png
    • 12.png


## 4. Nielsen's Heuristics and Output Format

In [6]:
HEURISTICS = [
    ("H1", "Visibility of System Status",
     "The design should always keep users informed about what is going on through appropriate feedback within a reasonable amount of time."),
    ("H2", "Match Between the System and the Real World",
     "The design should speak the users' language, using familiar words and concepts and following real-world conventions."),
    ("H3", "User Control and Freedom",
     "Users need a clearly marked emergency exit to leave an unwanted action or state."),
    ("H4", "Consistency and Standards",
     "Users should not have to wonder whether different words, situations, or actions mean the same thing."),
    ("H5", "Error Prevention",
     "A good design prevents problems from occurring by eliminating error-prone conditions."),
    ("H6", "Recognition Rather Than Recall",
     "Minimize users' memory load by making elements, actions, and options visible."),
    ("H7", "Flexibility and Efficiency of Use",
     "Shortcuts can accelerate interaction for experienced users while accommodating users with different levels of experience."),
    ("H8", "Aesthetic and Minimalist Design",
     "Interfaces should not contain irrelevant information that competes with relevant content."),
    ("H9", "Help Users Recognize, Diagnose, and Recover from Errors",
     "Error messages should use plain language, indicate the problem, and suggest a solution."),
    ("H10", "Help and Documentation",
     "The system should provide accessible documentation to help users complete their tasks."),
]

HEURISTICS_TEXT = "\n".join(
    f"{code} — {name}: {description}"
    for code, name, description in HEURISTICS
)

OUTPUT_FORMAT = '''Return ONLY a valid JSON array, with no additional text, Markdown, or code fences.
Each item must follow exactly this structure:
[
  {
    "violated_heuristic": "H[n] — [name]",
    "interface_element": "[description of the interface element]",
    "rationale": "[explanation of the usability violation]",
    "improvement_suggestion": "[recommended corrective action]",
    "severity": [0 to 4]
  }
]
Severity scale: 0 = cosmetic, 1 = minor, 2 = moderate, 3 = major, 4 = critical.
List only heuristics for which violations have been identified.'''

SYSTEM_PROMPT = (
    "You are a usability evaluation expert with knowledge of Nielsen's "
    "10 Usability Heuristics. Your task is to analyze digital interfaces "
    "and identify heuristic violations based on the provided content. "

    "FUNDAMENTAL CONSTRAINT: Analyze only what is visible and confirmed "
    "in the provided static screenshot. Do not infer, assume, or speculate "
    "about behaviors that would occur after user interactions such as clicks, "
    "typing, hovering, or navigation. "

    "Examples of what MUST NOT be evaluated: "
    "(1) how a form field behaves after the user starts typing; "
    "(2) whether a placeholder disappears when an input is clicked; "
    "(3) whether a menu expands on hover; "
    "(4) whether an error message appears after a form is submitted; "
    "(5) whether loading feedback appears after a button is clicked. "

    "Report a violation only when the problem is directly observable "
    "in the static image and does not depend on an interaction or a later "
    "interface state. If it is unclear whether a violation depends on "
    "dynamic interaction, omit it. "

    "Do not evaluate or comment on the language used in the interface; "
    "language is not an evaluation criterion. "
    "Return only the JSON array, with no comments outside the JSON."
)



## 5. Zero-Shot Prompting

In [7]:
def build_zero_shot_prompt(dom_content=None):
    """
    Zero-shot: includes only the task, the heuristics, and the interface content.
    It does not include examples or intermediate reasoning instructions.
    The screenshot is passed directly in the multimodal request.
    """
    parts = [
        "Perform a heuristic evaluation of the interface below "
        "using Nielsen's 10 Usability Heuristics.\n\n"
        "## Reference heuristics:\n" + HEURISTICS_TEXT
    ]

    if dom_content:
        parts.append("\n\n## Extracted interface HTML/DOM content:\n" + dom_content)
    # When no HTML is provided, the evaluation is based only on the screenshot.

    parts.append("\n\n## Output format:\n" + OUTPUT_FORMAT)
    return "".join(parts)




## 6. Few-Shot Prompting

In [8]:
FEW_SHOT_EXAMPLE = '''## Example of a heuristic evaluation

Interface under evaluation: an e-commerce product listing page.

Expected output:
[
  {
    "violated_heuristic": "H1 — Visibility of System Status",
    "interface_element": "Search results loading indicator",
    "rationale": "There is no visible indication that products are being loaded, leaving users without feedback while they wait.",
    "improvement_suggestion": "Add a spinner or progress bar while search results are loading.",
    "severity": 2
  },
  {
    "violated_heuristic": "H8 — Aesthetic and Minimalist Design",
    "interface_element": "Filter sidebar",
    "rationale": "The sidebar displays more than 20 filter options simultaneously, visually overwhelming users with information that may rarely be used.",
    "improvement_suggestion": "Collapse less frequently used filters into expandable groups and show only the most relevant filters by default.",
    "severity": 2
  }
]
'''


def build_few_shot_prompt(dom_content=None):
    """
    Few-shot: includes an annotated evaluation example before the actual task.
    """
    parts = [
        "Perform a heuristic evaluation of the interface below "
        "using Nielsen's 10 Usability Heuristics.\n\n"
        "## Reference heuristics:\n" + HEURISTICS_TEXT,
        "\n\n" + FEW_SHOT_EXAMPLE
    ]

    if dom_content:
        parts.append("\n\n## Extracted interface HTML/DOM content:\n" + dom_content)
    # When no HTML is provided, the evaluation is based only on the screenshot.

    parts.append("\n\n## Output format:\n" + OUTPUT_FORMAT)
    return "".join(parts)




## 7. Chain-of-Thought Prompting

In [9]:
def build_chain_of_thought_prompt(dom_content=None):
    """
    Chain-of-thought: asks the model to reason through the task in stages
    before producing the final output.
    Stages: (1) identify interface elements, (2) map them to heuristics,
    and (3) formulate the evaluation.
    """
    parts = [
        "Perform a heuristic evaluation of the interface below "
        "using Nielsen's 10 Usability Heuristics.\n\n"
        "## Reference heuristics:\n" + HEURISTICS_TEXT
    ]

    if dom_content:
        parts.append("\n\n## Extracted interface HTML/DOM content:\n" + dom_content)
    # When no HTML is provided, the evaluation is based only on the screenshot.

    parts.append(
        "\n\n## Step-by-step analysis instructions:\n"
        "Before producing the final output, internally consider the following steps:\n"
        "  1. Identify the visible interface elements (menus, buttons, forms, text, images, etc.).\n"
        "  2. For each element, consider potential violations of Nielsen's heuristics.\n"
        "  3. For each identified violation, formulate the rationale and an improvement suggestion.\n"
        "After completing this analysis, produce ONLY the final JSON array. "
        "Do not include intermediate reasoning in the response."
    )

    parts.append("\n\n## Output format:\n" + OUTPUT_FORMAT)
    return "".join(parts)




## 8. Model API Call

In [10]:
def call_gemini(prompt_text, image_base64=None):
    """Send a screenshot and prompt to Gemini."""

    contents = [
        types.Part.from_text(text=prompt_text)
    ]

    if image_base64:
        contents.append(
            types.Part.from_bytes(
                data=base64.b64decode(image_base64),
                mime_type="image/png"
            )
        )

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=2048,
            temperature=0
        )
    )

    return response.text

## 9. Evaluate the 12 UI Screenshots with GPT

In [11]:

import time
import json
import os

def run_evaluation(strategy, prompt_text, image_base64, max_retries=3):
    """Call Gemini, retry transient errors, and validate JSON output."""

    print(f"  ⏳ Strategy: {strategy} ...")

    for attempt in range(max_retries):
        try:
            raw_response = call_gemini(prompt_text, image_base64)

            if not raw_response:
                raise ValueError("Gemini returned an empty response.")

            # Clean Markdown code fences if present.
            cleaned_response = (
                raw_response.strip()
                .replace("```json", "")
                .replace("```", "")
                .strip()
            )

            # Parse and validate JSON.
            output = json.loads(cleaned_response)

            if not isinstance(output, list):
                raise ValueError("Expected a JSON array.")

            print(f"  ✅ Success: {len(output)} finding(s)")

            return {
                "status": "success",
                "findings": output,
                "error": None
            }

        except Exception as error:
            error_message = str(error)
            is_retryable = (
                "503" in error_message
                or "UNAVAILABLE" in error_message
                or "429" in error_message
                or "RESOURCE_EXHAUSTED" in error_message
            )

            if is_retryable and attempt < max_retries - 1:
                wait_time = 5 * (2 ** attempt)
                print(
                    f"  ⚠️ Temporary API error. "
                    f"Retry {attempt + 1}/{max_retries - 1} "
                    f"in {wait_time}s..."
                )
                time.sleep(wait_time)
                continue

            print(f"  ❌ Failed: {error_message}")

            return {
                "status": "failed",
                "findings": None,
                "error": error_message
            }

def autosave_results(all_results):
    os.makedirs("outputs", exist_ok=True)

    payload = {
        "model": GEMINI_MODEL,
        "total_uis": len(all_results),
        "results": all_results
    }

    with open(
        "outputs/evaluation_checkpoint.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print("💾 Checkpoint saved!")

# Main loop: iterate over every UI screenshot.
all_results = {}

STRATEGIES = {
    "Zero-shot": build_zero_shot_prompt,
    "Few-shot": build_few_shot_prompt,
    "Chain-of-thought": build_chain_of_thought_prompt,
}

total_evaluations = 0
successful_evaluations = 0
failed_evaluations = 0

for ui_index, screenshot_path in enumerate(ui_files, start=1):

    ui_name = os.path.splitext(os.path.basename(screenshot_path))[0]

    print(f"\n{'=' * 60}")
    print(f"🖼️ UI {ui_index}/{len(ui_files)}: {ui_name}")
    print(f"{'=' * 60}")

    # Prepare screenshot.
    resize_image(screenshot_path)
    image_base64 = image_to_base64(screenshot_path)

    all_results[ui_name] = {}
    autosave_results(all_results)
    time.sleep(3)

    for strategy, prompt_builder in STRATEGIES.items():

        prompt_text = prompt_builder(dom_content=None)

        result = run_evaluation(
            strategy,
            prompt_text,
            image_base64
        )

        all_results[ui_name][strategy] = result

        total_evaluations += 1

        if result["status"] == "success":
            successful_evaluations += 1
        else:
            failed_evaluations += 1


# Experiment execution summary.
print("\n" + "=" * 60)
print("🎯 EVALUATION SUMMARY")
print("=" * 60)

print(f"Total evaluations   : {total_evaluations}")
print(f"Successful          : {successful_evaluations}")
print(f"Failed              : {failed_evaluations}")

if total_evaluations:
    success_rate = successful_evaluations / total_evaluations * 100
    print(f"Success rate        : {success_rate:.2f}%")

print("=" * 60)



🖼️ UI 1/12: 01
[✓] Image is ready: 934x1600 px
💾 Checkpoint saved!


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


  ⏳ Strategy: Zero-shot ...
  ⚠️ Temporary API error. Retry 1/2 in 5s...
  ⚠️ Temporary API error. Retry 2/2 in 10s...
  ❌ Failed: 402 RESOURCE_EXHAUSTED. {'error': {'code': 402, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}
  ⏳ Strategy: Few-shot ...
  ⚠️ Temporary API error. Retry 1/2 in 5s...
  ⚠️ Temporary API error. Retry 2/2 in 10s...
  ❌ Failed: 402 RESOURCE_EXHAUSTED. {'error': {'code': 402, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}
  ⏳ Strategy: Chain-of-thought ...
  ⚠️ Temporary API error. Retry 1/2 in 5s...
  ⚠️ Temporary API error. Retry 2/2 in 10s...
  ❌ Failed: 402 RESOURCE_EXHAUS

## 10. Save Results as JSON

In [12]:
def save_all_results(all_results, batch_name="batch"):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"evaluation_{batch_name}_{timestamp}.json"

    payload = {
        "batch": batch_name,
        "timestamp": timestamp,
        "model": GEMINI_MODEL,
        "total_uis": len(all_results),
        "results": all_results,
    }

    with open(output_filename, "w", encoding="utf-8") as output_file:
        json.dump(payload, output_file, ensure_ascii=False, indent=2)

    print(f"[✓] Results saved to {output_filename}")
    return output_filename


output_file = save_all_results(all_results, batch_name="12_uis")

# Uncomment the following lines when running in Google Colab.
# from google.colab import files
# files.download(output_file)




[✓] Results saved to evaluation_12_uis_20261108_230407.json


## 11. Display Results

In [13]:
def display_results(all_results):
    for ui_name, strategies in all_results.items():
        print(f"\n{'#' * 60}")
        print(f"  🖼️  UI: {ui_name}")
        print(f"{'#' * 60}")

        for strategy, findings in strategies.items():
            print(f"\n  {'=' * 55}")
            print(f"  GPT | {strategy}")
            print(f"  {'=' * 55}")

            if not findings:
                print("  [No result or execution error]")
                continue

            for finding in findings:
                print(f"\n  🔴 {finding.get('violated_heuristic', '—')}")
                print(f"     Interface element : {finding.get('interface_element', '—')}")
                print(f"     Rationale          : {finding.get('rationale', '—')}")
                print(f"     Improvement        : {finding.get('improvement_suggestion', '—')}")
                print(f"     Severity           : {finding.get('severity', '—')}")


display_results(all_results)



############################################################
  🖼️  UI: 01
############################################################

  GPT | Zero-shot


AttributeError: 'str' object has no attribute 'get'